In [ ]:
%%capture
%run ../part_1/part1.ipynb
import math
from collections import Counter
from array import array
from collections import defaultdict

In [3]:
#We create a dictionary called field_terms for organizing the searchable terms separately for each field.
field_terms = {
    "title":       list(df["title_terms"]),
    "description": list(df["description_terms"]),
    "category":    list(df["category_terms"]),     
    "brand":       list(df["brand_terms"]),
    "seller":      list(df["seller_terms"]),
    "details":     [build_terms(details_to_text(x)) for x in df["product_details"]],   # values only
}

pids = list(df["pid"])
N = len(df)

Part 1: Indexing 

In [4]:
from collections import defaultdict
def create_index(all_terms):
    """
    Implement the inverted index of one field

    Argument:
    all_terms - list of words of all the term of all documents in that specific field

    Returns:
    index - the inverted index (implemented through a Python dictionary) containing terms as keys and the corresponding
    list of documents where these keys appears in (and the positions) as values.
    """
    index = defaultdict(list)

    for doc_id, terms in enumerate(all_terms):
        current_doc_index = {}  # Create temporary index for the current document

        for position, term in enumerate(terms):
            try:
                # if the term is already in the index for the current page (current_page_index)
                # append the position to the corresponding list (elemento 1 del arreglo, el 0 es la id del documento)
                current_doc_index[term][1].append(position)
            except:
                # Add the new term as dict key and initialize the array of positions and add the position
                current_doc_index[term] = [doc_id, array('I', [position])] #'I' indicates unsigned int (int in Python)

        #merge the current page index with the main index
        for term, posting in current_doc_index.items():
            index[term].append(posting)

    return index

In [5]:
# one inverted index per fiel
index = {}
for field in field_terms:
    index[field] = create_index(field_terms[field])
    print(field, ":", len(index[field]), "terms")

title : 738 terms
description : 4698 terms
category : 40 terms
brand : 402 terms
seller : 687 terms
details : 17812 terms


In [6]:
#We have decided the important of these fileds based on the result of part 1 validate labels and the result of the index
weights = {"title": 3.0,"description": 1.5,"category": 0.5,"brand": 1.0, "seller": 0.0, "details": 2.0}

In [7]:
def search(query, top_k=20):
    """
    The output is the list of documents that contain any of the query terms.
    So, we will get the list of documents for each query term, and take the union of them.
    """
    #Normalize the query, so it matches with the pre-processed documents and fields
    query_terms = list(set(build_terms(query)))
    if len(query_terms) == 0:
        return []

    #Find which documents contain each term

    tf = {} #frequency 
    term_docs = {}   # pid of the docuemtns that contain the word                   
    for term in query_terms:
        term_docs[term] = set()
        for field in index:
            tf[(field, term)] = {} #empty dictionary to store the frequency of the current word in the current field
            try:
                postings = index[field][term] # Get the postings for this term in this field
                for posting in postings:
                    doc_id = posting[0]
                    positions = posting[1]
                    tf[(field, term)][doc_id] = len(positions)
                    term_docs[term].add(doc_id)
            except KeyError: # This term does not exist in this field
                pass

        if not term_docs[term]:
            return []
        

    # AND, calculates the interseccion
    docs = None
    for term in query_terms:
        if len(term_docs[term]) == 0:
            return []
        if docs is None:
            docs = term_docs[term]
        else:
            docs = docs.intersection(term_docs[term])

    # TF-IDF score
    results = []
    for doc_id in docs:
        score = 0
        for term in query_terms:
            idf = math.log(N / len(term_docs[term]))
            for field in index:
                if doc_id in tf[(field, term)]:
                    score += weights[field] * (1 + math.log(tf[(field, term)][doc_id])) * idf
        results.append((doc_id, score))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

In [8]:
def show(query, top_k=10):
    """Print the top_k results of a query."""
    for doc_id, score in search(query, top_k):
        print(round(score, 3), pids[doc_id], df["title"].iat[doc_id], "|", df["brand"].iat[doc_id])

show("women full sleeve sweatshirt cotton")

46.436 SWSFFVK89VKGHBZN Full Sleeve Graphic Print Women Sweatshirt | CupidSto
46.436 SWSFFVK8RYHBBFAH Full Sleeve Graphic Print Women Sweatshirt | CupidSto
46.13 SWSFFVKBCQG5FHPF Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBH5YEGFFN Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBB8FHEB7R Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBCY7RSGYJ Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBQUWH72KH Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBYG9ZUBTW Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBF5U5AHBR Full Sleeve Printed Women Sweatshirt | CupidSto
45.184 SWSFWEF3XGHFBCJC Full Sleeve Graphic Print Women Sweatshirt | Free Authori


In [9]:
test_queries = {
    1: "men round neck cotton shirt",
    2: "women printed topwear",
    3: "men slim fit jeans",
    4: "women cotton dress",
    5: "men solid polo shirt",
}

for qid, q in test_queries.items():
    print(qid, q, "->", len(search(q, top_k=100000)), "documents")

1 men round neck cotton shirt -> 3227 documents
2 women printed topwear -> 3820 documents
3 men slim fit jeans -> 212 documents
4 women cotton dress -> 517 documents
5 men solid polo shirt -> 919 documents


**Part 2: Evaluation**

In [10]:
import numpy as np
import pandas as pd

1. Evaluation metrics

In [21]:
def precision_at_k(y_true, y_score, k=10):
    """P@K = (#relevant docs in the top K)/K"""
    y_score = np.asarray(y_score, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    order = np.argsort(-y_score, kind="stable") #stable so that documents with equal score keep the same order produced by search()
    top = np.take(y_true, order[:k])
    return float(np.sum(top) / k)

def recall_at_k(y_true, y_score, k=10, n_relevant=None):
    """R@K = (#relevant docs in the top K) / (#relevant docs for the query)
    n_relevant: total relevant docs known for the query (including the ones that the system did not retrieve)
    If None, the relevant docs inside y_true are used"""
    if n_relevant is None:
        n_relevant = np.sum(y_true)
    if n_relevant == 0:
        return 0.0
    y_score = np.asarray(y_score, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    order = np.argsort(-y_score, kind="stable") #stable so that documents with equal score keep the same order produced by search()
    top = np.take(y_true, order[:k])
    return float(np.sum(top)/n_relevant)

def f1_at_k(y_true, y_score, k=10, n_relevant=None):
    """F@K = harmonic mean of P@K and R@K"""
    p = precision_at_k(y_true, y_score, k)
    r = recall_at_k(y_true, y_score, k, n_relevant)
    if (p+r) == 0:
        return 0.0
    else:
        return float(2*p*r/(p+r))

def avg_precision_at_k(y_true, y_score, k=10):
    """
    Parameters
    ----------
    y_true: Ground truth (true relevance labels).
    y_score: Predicted scores.
    k : number of doc to consider.

    Returns
    -------
    average precision @k : float
    """

    y_score = np.asarray(y_score, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    order = np.argsort(-y_score, kind="stable") #stable so that documents with equal score keep the same order produced by search()
    
    prec_at_i = 0
    prec_at_i_list = []
    number_of_relevant = 0
    number_to_iterate = min(k, len(order))

    for i in range(number_to_iterate):
        if y_true[order[i]] == 1:
            number_of_relevant += 1
            prec_at_i = number_of_relevant / (i + 1)
            prec_at_i_list.append(prec_at_i)

    if number_of_relevant == 0:
        return 0.0
    else:
        return np.sum(prec_at_i_list) / number_of_relevant

def rr_at_k(y_true, y_score, k=10):
    """
    Parameters
    ----------
    y_true: Ground truth (true relevance labels).
    y_score: Predicted scores.
    k : number of doc to consider.

    Returns
    -------
    Reciprocal Rank for current query
    """

    y_score = np.asarray(y_score, dtype=float)
    y_true = np.asarray(y_true, dtype=float)

    order = np.argsort(-y_score, kind="stable")  # indexes sorted by descending score (stable: ties keep the order produced by search())
    y_true = np.take(y_true, order[:k])

    if np.sum(y_true)==0: # if there are not relevant doument return 0
        return 0
    return float(1.0/(np.argmax(y_true==1)+1)) # hint: to get the position of the first relevant document use "np.argmax"

def dcg_at_k(y_true, y_score, k=10):
    y_score = np.asarray(y_score, dtype=float)
    y_true = np.asarray(y_true, dtype=float)

    order = np.argsort(-y_score, kind="stable")  # indexes sorted by descending score (stable: ties keep the order produced by search())
    y_true = np.take(y_true, order[:k])
    gain = 2 ** y_true - 1  # Compute gain (use formula 7 above)
    discounts = np.log2(np.arange(len(y_true)) + 2)  # Compute denominator
    return float(np.sum(gain / discounts))  #return dcg@k

def ndcg_at_k(y_true, y_score, k=10, n_relevant=None):
    dcg_max = dcg_at_k(y_true, y_true, k)
    y_true = np.asarray(y_true, dtype=float)
    if n_relevant is None:
        dcg_max = dcg_at_k(y_true, y_true, k)              # ideal: the retrieved docs sorted by their true relevance
    else:
        ideal = np.ones(int(min(n_relevant, k)))           # ideal: all the known relevant docs first (binary labels)
        dcg_max = dcg_at_k(ideal, ideal, k)
    if dcg_max == 0:
        return 0.0
    return float(dcg_at_k(y_true, y_score, k) / dcg_max)

def map_at_k(search_res, k=10):
    '''
    Parameters
    ----------
    search_res: search results dataset containing:
        q_id: query id.
        doc_id: document id.
        predicted_relevance: relevance predicted through LightGBM.
        y_true: actual score of the document for the query (ground truth).

    Returns
    -------
    mean average precision @k : float
    '''
    avp = []
    for q in search_res["query_id"].unique():  #loop over all query id
        curr_data = search_res[search_res["query_id"] == q]  # select data for current query
        avp.append(avg_precision_at_k(np.array(curr_data["bin_y_true"]), np.array(curr_data["predicted_relevance"]),
                                      k))  #append average precision for current query
    return float(np.sum(avp) / len(avp)), avp  # return mean average precision

def mrr_at_k(per_query_rr):
    """MRR = mean of the RR@K of every query."""
    return float(np.mean(per_query_rr))



Sanity checks as in the labs + comparison with scikit-learn

In [23]:
from sklearn.metrics import average_precision_score, ndcg_score

# AP@K: lab check
y_true = np.array([1, 1, 0, 1, 0, 0, 1]); y_scores = np.array([7, 6, 5, 4, 3, 2, 1])
assert abs(average_precision_score(y_true[:3], y_scores[:3]) - avg_precision_at_k(y_true, y_scores, 3)) < 1e-12

# AP@K: manual computation (relevant docs at ranks 1,2,5,7,8,9)
yt = np.zeros(10); yt[[0, 1, 4, 6, 7, 8]] = 1
assert abs(avg_precision_at_k(yt, np.arange(10, 0, -1), 10) - (1 + 1 + 3/5 + 4/7 + 5/8 + 6/9) / 6) < 1e-12

# RR@K: first relevant doc is at rank 2 -> 0.5
assert rr_at_k([0, 1, 0, 1, 1], [.9, .5, .6, .7, .2], 5) == 0.5

# NDCG@K (binary labels, all docs retrieved) vs scikit-learn
rng = np.random.default_rng(0)
for _ in range(50):
    yt, ys = rng.integers(0, 2, 30), rng.random(30)
    if yt.sum() > 0:
        assert abs(ndcg_at_k(yt, ys, 10) - ndcg_score([yt], [ys], k=10)) < 1e-9
print("All metric checks passed")

All metric checks passed


2. Evaluation on the predefined queries

In [24]:
labels_df = pd.read_csv("../../data/validation_labels.csv")
validation_queries = {
    1: "women full sleeve sweatshirt cotton",
    2: "men slim jeans blue"
}

print(labels_df.groupby(["query_id", "labels"]).size())

query_id  labels
1         0          7
          1         13
2         0         10
          1         10
dtype: int64


In [34]:
def get_evaluation_data(query_id, query, k=10):
    """function for comparing the results of our search function
    with the relevance labels"""

    #select the labels of the query we are evaluating
    query_labels = labels_df[labels_df["query_id"]==query_id]

    #save the pids evaluated for this query (relevant and non relevant)
    judged_pids = set(query_labels["pid"])

    #save the pids that are relevant for the query
    relevant_pids = set(query_labels[query_labels["labels"]==1]["pid"])

    #execute search() to evaluate ou search system
    results = search(query, top_k=len(pids)) #top_=len(pids) since we want to retrieve all the possible results before we filter the ones that are evaluated

    #obtain the pids of the top k results since results has docs ids
    retrieved_pids = [pids[doc_id] for doc_id, score in results]

    #we only want the evaluated products
    evaluated_pids = []

    for pid in retrieved_pids: #iterate through all retrieved products by the search function
        if pid in judged_pids: #if is in the evaluated products in the csv
            evaluated_pids.append(pid) #append it into evaluated pids

    evaluated_pids = evaluated_pids[:k] #only the top k evaluated (in the same order as the search system)

    y_true = [] #relevance list

    for pid in evaluated_pids: #iterate through all the evaluated products
        if pid in relevant_pids: # if is in the relevant products, add 1; else, add 0
            y_true.append(1)
        else:
            y_true.append(0)

    total_relevant = len(relevant_pids) #total of relevant products

    return y_true, total_relevant, evaluated_pids





We prove the function

In [33]:
for query_id, query in validation_queries.items():

    y_true, total_relevant, evaluated_pids = get_evaluation_data(
        query_id, query, k=10
    )

    print("Query:", query)
    print("Relevance labels:", y_true)
    print("Evaluated products retrieved:", len(evaluated_pids))
    print("Total relevant:", total_relevant)
    print()

Query: women full sleeve sweatshirt cotton
Relevance labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Evaluated products retrieved: 10
Total relevant: 13

Query: men slim jeans blue
Relevance labels: [1, 1, 1, 0, 1, 1, 1, 0]
Evaluated products retrieved: 8
Total relevant: 10

